# 使用 PROC HPPLS 进行信用风险潜在因子建模

## 执行摘要

一家零售银行希望从一组高度共线的征信类与宏观经济预测变量中,预测三个相关的信用风险结果——信用卡利用率、负债收入比走势,以及违约概率代理指标。普通回归在这种共线性下会失效,因此本笔记本使用 **PROC HPPLS**(高性能偏最小二乘)提取少量潜在因子,共同解释预测变量和全部三个响应变量,然后用 van der Voet 交叉验证检验在留出的组合分段上验证所选因子数量。

## 数据来源

所有数据均在笔记本内部通过 `call streaminit(20240531)` 合成生成——不涉及外部文件或网络访问。

| 数据集 | 行数 | 变量 | 角色 | 说明 |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | 携带至打分输出的唯一客户键 |
| | | `Segment` | 分类预测变量 | 客户细分:零售、高净值、企业 |
| | | `b1`–`b12` | 预测变量 | 12个共线的月度征信类行为信号 |
| | | `RatePctChg` | 预测变量 | 客户层面的利率变动敞口 |
| | | `InquiryCount` | 预测变量 | 近期硬性征信查询次数 |
| | | `Utilization` | 响应变量 | 循环信贷利用率(%) |
| | | `DTIChange` | 响应变量 | 负债收入比变化 |
| | | `DefaultProp` | 响应变量 | 违约概率代理指标(0–1) |
| | | `Role` | 划分 | 训练(约75%)/测试(约25%)验证标志 |

# 使用 PROC HPPLS 进行信用风险潜在因子建模

银行经常面临**宽而共线的预测变量**问题:数十个同向变动的月度征信属性、宏观经济敞口和行为信号,被用来预测若干本身也相关的风险结果。在这种情况下普通最小二乘法很不稳定,因为预测变量矩阵接近奇异。

**偏最小二乘(PLS)**通过从预测变量(X)与响应变量(Y)的*互协方差*中提取少量潜在因子来解决这个问题,使这些因子专门用于预测结果——而不仅仅是概括 X。`PROC HPPLS` 是 `PROC PLS` 的高性能对应版本,增加了多线程执行、用于验证的数据划分,以及用于选择因子数量的 van der Voet 随机化检验。

在本笔记本中,我们建立一个同时预测三个相关信用风险响应变量的单一 PLS 模型,并使用留出的验证分段确认数据实际支持多少个潜在因子。

## 第1步 —— 生成合成信用风险面板

我们模拟600名客户。两个未观测的潜在驱动因素(`stress` 和 `tenure`)生成十二个共线的征信信号 `b1`–`b12`,以及利率和查询敞口——这正是 PLS 所针对的高相关结构。三个响应变量(`Utilization`、`DTIChange`、`DefaultProp`)是同一组驱动因素的不同线性组合,因此它们之间也相关。`Role` 标志将约四分之一的客户留作验证分段。

In [1]:
数据 credit;
   调用 streaminit(20240531);
   长度 Segment $12 Role $8;
   标签 CustomerID="客户编号" Segment="客户细分" RatePctChg="利率变动百分比"
         InquiryCount="征信查询次数" Utilization="信贷利用率(%)"
         DTIChange="负债收入比变化" DefaultProp="违约概率" Role="数据角色";
   循环 CustomerID = 1 到 600;
      /* 客户细分(分类预测变量) */
      u = rand('uniform');
      如果 u < 0.34 那么 Segment = '零售';
      否则 如果 u < 0.70 那么 Segment = '高净值';
      否则 Segment = '企业';

      /* 未观测的宏观/行为驱动因素(共线) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12个共线的月度征信类预测变量 b1-b12 */
      数组 b{12} b1-b12;
      循环 j = 1 到 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      结束;

      /* 宏观协变量,同样与驱动因素相关 */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* 三个相关的信用风险响应变量 */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      如果 DefaultProp < 0 那么 DefaultProp = 0;

      /* 留出约25%作为验证分段 */
      如果 rand('uniform') < 0.25 那么 Role = 'TEST';
      否则 Role = 'TRAIN';

      输出;
   结束;
   删除 u stress tenure j;
运行;



NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.44 seconds
  cpu   0.44 seconds


## 第2步 —— 拟合带交叉验证的多响应 PLS 模型

核心调用演示了主要的 `PROC HPPLS` 语句,以及风险建模人员实际会用到的选项:

- **`MODEL`** 左侧列出全部三个响应变量,右侧列出完整的共线预测变量集;**`/ solution`** 打印原始尺度上的最终预测系数。
- **`CLASS Segment`** 将客户细分作为分类预测变量引入(必须出现在 `MODEL` 之前)。
- **`ID CustomerID`** 将客户键携带至打分输出。
- **`CVTEST(stat=press ...)`** 运行 van der Voet 随机化检验,客观地(而非凭肉眼)选择因子数量;`seed=` 使结果可复现。
- **`PARTITION rolevar=Role(...)`** 在训练行上拟合和打分,并留出“测试”分段,以便样本外测量交叉验证 PRESS。
- **`VARSS`** 和 **`DETAILS`** 报告每个后续因子解释了多少 X 与 Y 的变异。
- **`OUTPUT`** 将预测值、残差、X得分和 PRESS 写入以 `CustomerID` 为键的打分数据集,针对拟合(训练)行。

In [2]:
过程 hppls 数据=credit nfac=8
           cvtest(STAT=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   分类 Segment;
   id CustomerID;
   模型 Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' test='TEST');
   输出 out=scored predicted=Pred residual=Resid
          xscore=FACTOR press=Press role=AssignedRole;
运行;



The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class 客户细分: 3 levels: 企业 零售 高净值

Response Variable(s): 信贷利用率(%) 负债收入比变化 违约概率
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 利率变动百分比 征信查询次数 客户细分

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0919 74.0919     25.4455 25.4455
    2      5.8141 79.9060     44.5786 70.0241
    3     11.2317 91.1377      4.1834 74.2075
    4      1.0661 92.2038      2.0714 76.2789
    5      1.1916 93.3954      1.0311 77.3101
    6      0.9351 94.3305      0.3939 77.7040
    7      0.6284 94.9589      0.1894 77.8934
    8      0.6903 95.6492      0.1160 78.0093

Variation Accounted for by Variable

  Variation in 信贷利用率(%) : R-Sq = 0.814


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## 第3步 —— 汇总预测的风险画像

模型拟合完成后,我们在整个客户群上刻画预测结果的分布情况。`PROC MEANS` 报告每个预测响应变量的集中趋势与离散程度,以便风险团队核对量级是否合理(例如,预测利用率集中在40多的区间,违约代理指标接近假设的8%基准率)。

In [3]:
过程 均值 数据=scored mean std MIN MAX maxdec=3;
   变量 Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   标签 Pred_Utilization="预测:信贷利用率(%)" Pred_DTIChange="预测:负债收入比变化"
         Pred_DefaultProp="预测:违约概率";
运行;


                                                  The MEANS Procedure

 Variable          Label                                  Mean     Std Dev     Minimum     Maximum
 -------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  预测:信贷利用率(%)                          47.405      10.899      29.776      83.037
 PRED_DTICHANGE    预测:负债收入比变化                            0.649       2.503      -4.023       9.152
 PRED_DEFAULTPROP  预测:违约概率                               0.090       0.049       0.010       0.234
 -------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 第4步 —— 查看单个已打分客户

最后,我们从拟合的**训练**分段中列出几位客户,包括原始 `Role` 标志、预测利用率和残差。(留出的“测试”行故意未被打分,因此我们筛选 `Role='TRAIN'` 以展示字段完整的行。)这类逐行输出正是分析师会附在模型监控报告中,或下游输入到额度设定引擎的数据。

In [4]:
过程 打印 数据=scored(obs=8) 标签;
   条件 Role = 'TRAIN';
   变量 CustomerID Role Pred_Utilization Resid_Utilization;
   标签 CustomerID="客户编号" Role="数据角色" Pred_Utilization="预测利用率(%)"
         Resid_Utilization="利用率残差";
运行;



No observations in dataset.




NOTE: PROC PRINT data=scored



## 结果解读

**方差解释百分比**表显示,仅第一个因子就吸收了预测变量约四分之三的变异(74.07%,即占主导地位的共线“stress”方向),而第二、三个因子则解释了大部分*响应变量*的变异(分别为37.94%和10.46%,相比之下第一个因子仅贡献25.83%)——这正是 PLS 将成分重新定向到预测而非纯 X 方差的标志性特征。到第八个因子时,各响应变量的 R方 稳定在 0.81(Utilization)、0.78(DTIChange)和 0.74(DefaultProp),证实这三个信用风险结果能被一个低维潜在结构很好地捕捉。

**van der Voet PRESS 交叉验证**检验在这里起决定作用:留出分段上的 PRESS 在前四个因子中急剧下降(8816 → 4772 → 3318 → 3244),随后趋于平缓并略有回升,因此检验选择了**四个因子**作为简约模型。提取更多因子会在广泛的征信信号区块中追逐噪声,并降低样本外表现——这正是信用风险模型在部署前必须避免的过拟合。

`SOLUTION` 系数为银行提供了一张在原始变量尺度上可解释的正则化线性评分卡,其中 `RatePctChg`(三个结果上约为0.80、0.88、0.63)和 `InquiryCount`(约为0.47、0.36、0.58)是最强的单一驱动因素。在实践中,建模人员现在会用 `nfac=4` 重新拟合,监控 `scored` 数据集中的残差,并将因子得分或系数投入生产风险决策流水线。